# SIGNOVA: Cloud-Native Machine Learning Starter
### Direct Streaming & Training on How2Sign & ArabSign without Local Storage

This notebook enables running sign language ML experiments on **Google Colab** or **Kaggle** utilizing remote Google Drive streams, eliminating the need to download 2+ TB of video files onto your personal computer.

In [ ]:
# 1. Environment & Dependencies Setup
!pip install -q gdown torch torchvision opencv-python pandas

## 2. Resource Direct Access Identifiers
Below are the verified direct Google Drive IDs and download targets for both continuous datasets.

In [ ]:
DATASET_LINKS = {
    'how2sign': {
        'val_rgb_clips': '1oVZyTWhHShyqshC2kUrfWnBF8apIR7Z1',       # ~7 GB
        'test_rgb_clips': '1d6GHqu0_8IGiKbu3sTZHtMb0DGhbHSMu',      # ~11 GB
        'val_2d_keypoints': '1aOhRknNWj8APdxHmwJdQrMo5xuIGNXxM',    # ~3.8 GB
        'test_2d_keypoints': '1quj8Ipm56pH65KAKK3Pc-sqZ0ozw2gSe',   # ~5.8 GB
    },
    'arabsign': {
        'code_and_splits_zip': 'https://github.com/Hamzah-Luqman/ArabSign/raw/main/ArabSign-EncoderDec.zip',
        'skeletons_folder_gdrive_id': '18QN4X2vnWZ5PvNVik6TJJ7N606pRMvwA', # ~5 GB
        'rgb_videos_folder_gdrive_id': '1twRzL8fqjbq5dp6jV-V00-O8gD1BZMTN', # ~60 GB
    }
}

print('Cloud dataset links ready.')

## 3. Streaming Remote Video Streamer (PyTorch IterableDataset)
This dataset reads video frames on the fly without saving video files to hard disk.

In [ ]:
import cv2
import torch
from torch.utils.data import IterableDataset, DataLoader

class CloudVideoStreamer(IterableDataset):
    """
    Streams video frames directly over network into GPU/RAM.
    Discards frames immediately after forward pass to save storage.
    """
    def __init__(self, video_urls, target_frames=32, target_size=(224, 224)):
        self.video_urls = video_urls
        self.target_frames = target_frames
        self.target_size = target_size

    def __iter__(self):
        for url in self.video_urls:
            cap = cv2.VideoCapture(url)
            frames = []
            while cap.isOpened() and len(frames) < self.target_frames:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.resize(frame, self.target_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            cap.release()
            
            if len(frames) > 0:
                # Output tensor format: [C, T, H, W]
                tensor = torch.tensor(frames).permute(3, 0, 1, 2).float() / 255.0
                yield tensor

print('CloudVideoStreamer class compiled successfully.')